# Workflow tiếng Việt cho Checkout the Canteen

Notebook này là bản hướng dẫn chính sau khi đã dọn project. Chạy cell theo nhu cầu, không cần chạy hết từ đầu tới cuối.

## 0. Đưa notebook về đúng thư mục project

VSCode đôi khi chạy notebook với working directory là `notebooks/`, làm lệnh `scripts/...` bị hiểu nhầm thành `notebooks/scripts/...`. Hãy chạy cell này trước.

In [ ]:
from pathlib import Path
import os, sys

cwd = Path.cwd().resolve()
project_root = cwd
if not (project_root / 'scripts').exists():
    for parent in [cwd.parent, *cwd.parents]:
        if (parent / 'scripts').exists() and (parent / 'prices.csv').exists():
            project_root = parent
            break

os.chdir(project_root)
print('PROJECT_ROOT =', project_root)
print('Python =', sys.executable)

## 1. Kiểm tra nhanh Python/GPU

Không còn dùng smoke-test script riêng. Cell này chỉ kiểm tra môi trường hiện tại có import được PyTorch và thấy GPU không.

In [ ]:
import torch, sys
print(sys.version)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Audit data chính thức

`scripts/20_audit_dataset_conflicts.py` dùng để tìm ảnh lỗi, exact duplicate, và ảnh gần giống pHash nhưng nằm ở class khác.

In [ ]:
!{sys.executable} scripts/20_audit_dataset_conflicts.py --root data/classification --phash-threshold 4

## 3. Mở Data IDE

`scripts/21_data_ide.py` là app chính để lọc data.

- Chọn `classification` thì IDE hiện `split` và `class`.
- Chọn `external_review`, `external_reviewed`, hoặc `quarantine` thì IDE hiện `Folder / pool`.
- Sau khi move/quarantine/future-use, ảnh đã xử lý rời khỏi queue và IDE nạp ảnh tiếp theo.
- Phím nhanh: `1-0` chọn ảnh trong hàng, `A` chọn cả hàng, `Space` xuống hàng, `Enter` move, `Q` quarantine, `F` future-use, `P` predict.

In [ ]:
!{sys.executable} scripts/21_data_ide.py --host 127.0.0.1 --port 7862

## 4. Tìm dataset public trước khi crawl

`scripts/22_search_public_datasets.py` tìm link Kaggle/GitHub/Roboflow/HuggingFace liên quan đến món cần bổ sung.

In [ ]:
!{sys.executable} scripts/22_search_public_datasets.py --provider mixed --max-results 10

## 5. Crawl ảnh candidate khi thiếu data

`scripts/09_collect_web_images.py` chỉ tải ảnh vào raw batch. Không đưa thẳng vào train.

In [ ]:
!{sys.executable} scripts/09_collect_web_images.py --queries configs/green_vegetable_extra_queries.csv --provider mixed --out data/downloads/scrape_batches/new_batch/raw --manifest data/downloads/scrape_batches/new_batch/scraped_manifest.csv --per-query 80 --max-downloads-per-class 300 --dedupe-against data/classification --dedupe-against data/downloads --phash-threshold 6

## 6. Import/preprocess crawl batch vào review

`scripts/18_import_scrape_batch_to_review.py` chuẩn hóa ảnh, lọc ảnh lỗi/mờ/trùng, rồi đưa vào folder review hoặc reviewed.

In [ ]:
!{sys.executable} scripts/18_import_scrape_batch_to_review.py --source data/downloads/scrape_batches/new_batch/raw --class-name rau_xao --pool rau_xao_new_batch --target review --dedupe-against data/classification

## 7. Import external datasets đã tải

`scripts/14_import_external_datasets.py` đọc các dataset tải sẵn trong `data/downloads/`, crop/normalize nếu cần, rồi đưa vào `external_staging/.../review`.

In [ ]:
!{sys.executable} scripts/14_import_external_datasets.py

## 8. Build lại data/classification

`scripts/17_build_weighted_classification_dataset.py` merge nguồn old và reviewed. Hiện tại hai nguồn được tin ngang nhau: weight 1 và weight 1.

In [ ]:
!{sys.executable} scripts/17_build_weighted_classification_dataset.py --old-weight 1 --reviewed-weight 1 --cross-class-hamming 4 --clear

## 9. Train model

`scripts/05_train_classifier.py` train classifier 11 món. Script lưu checkpoint tốt nhất theo validation.

In [ ]:
!{sys.executable} scripts/05_train_classifier.py --arch efficientnet_b0 --epochs 6 --batch-size 8 --lr 0.0001 --label-smoothing 0.05

## 10. Đọc report train

In [ ]:
!type outputs/reports/classification_report.txt

## 11. Demo checkout

`scripts/19_demo_checkout_app.py` là app demo hóa đơn: upload/chọn ảnh khay, chỉnh crop, ignore vùng không tính tiền, chạy classifier và xuất bill.

In [ ]:
!{sys.executable} scripts/19_demo_checkout_app.py --host 127.0.0.1 --port 7861